# Piper voice clone on Google Colab

Clone your voice with **Qwen3-TTS** (dataset) + **Piper** fine-tuning (ONNX model).

Run the cells top to bottom — all output appears inline in the notebook.

All heavy work uses **local Colab disk** (`/content/piper-work`). Google Drive receives the dataset zip, training checkpoints, and ONNX models when publish runs.

## Recommendations

- **GPU:** **L4** is the best value on Colab Pro (faster than T4, much cheaper than A100).
- **Dataset size:** default **10** utterances in cell 10 to verify voice quality — change to **2,000–2,500** before training.
- **Training time (≈2,000 samples on L4):** epoch 1 ~**30 min** (cache warmup), then ~**1 min/epoch**.

**Before you start:** Runtime → **Change runtime type** → **GPU** → **L4** (or T4 / A100).

Repo: [smart-puppet/piper-voice-clone](https://github.com/smart-puppet/piper-voice-clone)

In [ ]:
# @title 0. GPU check
# Recommended: L4 (best value on Colab Pro). T4 works but is slower; A100 is fastest but costly.
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → L4 (recommended) or T4 / A100")
name = torch.cuda.get_device_name(0)
print("GPU:", name)
if "L4" not in name and "A100" not in name:
    print("Tip: L4 is the best price/performance choice for this pipeline.")

In [ ]:
# @title 1. Clone project
from pathlib import Path

ROOT = Path("/content/piper-voice-clone")
REPO_URL = "https://github.com/smart-puppet/piper-voice-clone.git"

if (ROOT / "pyproject.toml").is_file():
    print("Project already present:", ROOT)
else:
    print("Cloning", REPO_URL)
    !git clone --branch main --depth 1 {REPO_URL} /content/piper-voice-clone
    print("Cloned to", ROOT)

%cd /content/piper-voice-clone

In [ ]:
# @title 2. Install (once per runtime)
from pathlib import Path

ROOT = Path("/content/piper-voice-clone")
marker = ROOT / ".colab_install_done"

if marker.is_file():
    print("Install already completed this session — delete", marker, "to re-run.")
else:
    print("Running install/install_colab.sh...")
    %cd /content/piper-voice-clone
    !bash install/install_colab.sh
    marker.write_text("ok\n")
    print("Install complete.")

In [ ]:
# @title 3. Mount Google Drive (required before publish)
from google.colab import drive

drive.mount("/content/drive")

### Project settings (cell 4)

Writes **`config.colab.yaml`**. Set the form below, then run the code cell.

**`LANGUAGE_CODE`, checkpoint, and book text must use the same language** (e.g. all `de`).

| Parameter | What it does |
|-----------|--------------|
| **VOICE_NAME** | Short label for your clone (`myvoice-de_v01`, `myvoice-de-medium.onnx`). |
| **LANGUAGE_CODE** | Dataset + Piper language: `de`, `en`, or `fr`. |
| **DATASET_VERSION** | Dataset folder suffix (e.g. `v01`). Bump when regenerating. |
| **CHECKPOINT_PRESET** | Pretrained Piper voice to fine-tune from (downloaded in cell 8). |
| **CUSTOM_CHECKPOINT_HF_PATH** | When preset is **custom**: path inside [rhasspy/piper-checkpoints](https://huggingface.co/datasets/rhasspy/piper-checkpoints). |
| **BOOK_PRESET** | Fairy-tale sentences for the dataset (Project Gutenberg). |
| **CUSTOM_BOOK_URL** | When preset is **custom**: Gutenberg plain-text URL. |

**Checkpoint presets** (medium):

| Preset | Language | Voice |
|--------|----------|-------|
| `de-thorsten-emotional-medium` | German | Thorsten expressive **(recommended)** |
| `de-thorsten-medium` | German | Thorsten neutral |
| `en-lessac-medium` | English | Lessac |
| `en-amy-medium` | English | Amy |
| `fr-siwis-medium` | French | Siwis (neutral) |
| `fr-tom-medium` | French | Tom (warmer) |
| `custom` | any | Your own HF path |

**Book presets:**

| Preset | Language | Book |
|--------|----------|------|
| `de-andersen-maerchen` | German | Andersen, *Märchen* |
| `en-grimm-fairy-tales` | English | Grimm, *Household Tales* |
| `fr-perrault-contes` | French | Perrault, *Contes de ma mère l'Oye* |
| `custom` | any | Your own Gutenberg URL |

In [ ]:
# @title 4. Project settings
VOICE_NAME = "myvoice"  # @param {type:"string"}
LANGUAGE_CODE = "de"  # @param ["de", "en", "fr"] {type:"string"}
DATASET_VERSION = "v01"  # @param {type:"string"}
CHECKPOINT_PRESET = "de-thorsten-emotional-medium"  # @param ["de-thorsten-emotional-medium", "de-thorsten-medium", "en-lessac-medium", "en-amy-medium", "fr-siwis-medium", "fr-tom-medium", "custom"] {type:"string"}
CUSTOM_CHECKPOINT_HF_PATH = ""  # @param {type:"string"}
BOOK_PRESET = "de-andersen-maerchen"  # @param ["de-andersen-maerchen", "en-grimm-fairy-tales", "fr-perrault-contes", "custom"] {type:"string"}
CUSTOM_BOOK_URL = ""  # @param {type:"string"}

from pathlib import Path

import sys
import yaml

ROOT = Path("/content/piper-voice-clone")
sys.path.insert(0, str(ROOT / "src"))
from voice_cloning.book_loader import resolve_book_preset, resolve_book_source
from voice_cloning.checkpoint_download import resolve_checkpoint_download

CONFIG_PATH = ROOT / "config.colab.yaml"

_, EFFECTIVE_CHECKPOINT_HF_PATH = resolve_checkpoint_download(
    preset=CHECKPOINT_PRESET,
    hf_path=CUSTOM_CHECKPOINT_HF_PATH,
)
BOOK_SOURCE, _ = resolve_book_source(preset=BOOK_PRESET, custom_url=CUSTOM_BOOK_URL)
TTS_LANGUAGE = {"de": "German", "en": "English", "fr": "French"}[LANGUAGE_CODE]

with CONFIG_PATH.open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config.update(
    {
        "voice_name": VOICE_NAME,
        "language_code": LANGUAGE_CODE,
        "dataset_version": DATASET_VERSION,
        "espeak_voice": LANGUAGE_CODE,
        "book_source": BOOK_SOURCE,
    }
)
config.setdefault("tts", {})
config["tts"]["language"] = TTS_LANGUAGE

with CONFIG_PATH.open("w", encoding="utf-8") as handle:
    yaml.dump(config, handle, default_flow_style=False, sort_keys=False, allow_unicode=True)

DATASET_NAME = f"{VOICE_NAME}-{LANGUAGE_CODE}_{DATASET_VERSION}"
print("Updated", CONFIG_PATH)
print("Dataset :", DATASET_NAME)
if CHECKPOINT_PRESET == "custom":
    print("Checkpoint:", "custom →", EFFECTIVE_CHECKPOINT_HF_PATH)
else:
    print("Checkpoint:", CHECKPOINT_PRESET)
if BOOK_PRESET == "custom":
    print("Book      :", "custom URL")
else:
    book = resolve_book_preset(BOOK_PRESET)
    print("Book      :", f"{book.author} — {book.title} ({BOOK_PRESET})")
print("Book URL  :", BOOK_SOURCE)
if CHECKPOINT_PRESET != "custom" and not CHECKPOINT_PRESET.startswith(LANGUAGE_CODE + "-"):
    print("Warning: CHECKPOINT_PRESET", CHECKPOINT_PRESET, "may not match LANGUAGE_CODE", LANGUAGE_CODE)
if CHECKPOINT_PRESET == "custom" and not EFFECTIVE_CHECKPOINT_HF_PATH.startswith(LANGUAGE_CODE + "/"):
    print("Warning: CUSTOM_CHECKPOINT_HF_PATH may not match LANGUAGE_CODE", LANGUAGE_CODE)
if BOOK_PRESET != "custom":
    book = resolve_book_preset(BOOK_PRESET)
    if book.lang != LANGUAGE_CODE:
        print("Warning: BOOK_PRESET", BOOK_PRESET, "is", book.lang, "but LANGUAGE_CODE is", LANGUAGE_CODE)

In [ ]:
# @title 5. Upload reference voice
# Mono WAV, 3–15 seconds, clean speech + exact transcript in .txt
from pathlib import Path

from google.colab import files

ROOT = Path("/content/piper-voice-clone")
REF_DIR = ROOT / "references"
REF_DIR.mkdir(parents=True, exist_ok=True)

print("1/2 — Select reference.wav (your voice clip)")
for name, data in files.upload().items():
    if not name.lower().endswith(".wav"):
        print(f"Skipping {name} (expected .wav)")
        continue
    wav_path = REF_DIR / "reference.wav"
    wav_path.write_bytes(data)
    print(f"Saved {wav_path} ({len(data) // 1024} KiB)")

print("\n2/2 — Select reference.txt (exact transcript of the WAV)")
for name, data in files.upload().items():
    if not name.lower().endswith(".txt"):
        print(f"Skipping {name} (expected .txt)")
        continue
    txt_path = REF_DIR / "reference.txt"
    txt_path.write_bytes(data)
    print(f"Saved {txt_path}")
    print("Transcript:", data.decode("utf-8", errors="replace").strip())

In [ ]:
# @title 6. Verify setup
import sys
from pathlib import Path

import torch

ROOT = Path("/content/piper-voice-clone")
sys.path.insert(0, str(ROOT / "src"))

from voice_cloning.dataset_generator import load_config

config = load_config(ROOT / "config.colab.yaml")
ref_wav = config.reference.wav_path
ref_txt = ROOT / "references" / "reference.txt"

errors = []
if not torch.cuda.is_available():
    errors.append("GPU not enabled (Runtime → Change runtime type → L4)")
if not (ROOT / ".colab_install_done").is_file():
    errors.append("install not completed (run cell 2)")
if not ref_wav.is_file():
    errors.append(f"missing reference WAV: {ref_wav}")
if not ref_txt.is_file():
    errors.append(f"missing reference transcript: {ref_txt}")

if errors:
    raise RuntimeError("\n".join(errors))

print("Setup OK —", f"{config.voice_name}-{config.language_code}_{config.dataset_version}")

In [ ]:
# @title 7. Anti-disconnect (run before dataset generation or training)
# Clicks Colab "Connect" every 30s while long notebook cells run.
from IPython.display import Javascript, display

display(
    Javascript(
        """
function piperClickConnect() {
  const host = document.querySelector("colab-connect-button");
  if (host && host.shadowRoot) {
    const btn = host.shadowRoot.querySelector("#connect");
    if (btn) { btn.click(); return; }
  }
  const legacy = document.querySelector("colab-toolbar-button#connect");
  if (legacy) legacy.click();
}
console.log("Anti-disconnect active (every 30s)");
setInterval(piperClickConnect, 30000);
"""
    )
)
print("Anti-disconnect running — keep this browser tab open during dataset generation and training.")

In [ ]:
# @title 8. Download pretrained Piper checkpoint
# Fetches the checkpoint from Hugging Face (rhasspy/piper-checkpoints).
# Saves weights to /content/piper-work/checkpoints/pretrained-finetune.ckpt for training.
# Re-run cell 4 first if you changed checkpoint or language settings.
%cd /content/piper-voice-clone
if CHECKPOINT_PRESET == "custom":
    !bash colab/download_checkpoint.sh --path {EFFECTIVE_CHECKPOINT_HF_PATH}
else:
    !bash colab/download_checkpoint.sh --preset {CHECKPOINT_PRESET}

### Import dataset (cell 9)

Alternative to **cell 10 (Generate dataset)**. Imports a Piper dataset zip (`metadata.csv` + `wavs/`) from **Google Drive** into:

`/content/piper-work/datasets/{voice}-{lang}_{version}/`

| Parameter | What it does |
|-----------|--------------|
| **IMPORT_FROM_DRIVE** | Checked: extract a dataset zip from Drive. Unchecked (default): skip — generate in cell 10 instead. |
| **DRIVE_ZIP_PATH** | Path to `.zip` on Drive. Leave empty for default `MyDrive/piper-voice-clone/archives/{dataset}.zip`. |
| **FORCE_IMPORT** | Checked: replace an existing dataset / overwrite WAVs when merging partial archives. |

Zip layout must match this project (full archive with metadata + wavs, or partial WAV merge into existing dataset).

In [ ]:
# @title 9. Import dataset (optional)
IMPORT_FROM_DRIVE = False  # @param {type:"boolean"}
DRIVE_ZIP_PATH = ""  # @param {type:"string"}
FORCE_IMPORT = False  # @param {type:"boolean"}

import sys
from pathlib import Path

ROOT = Path("/content/piper-voice-clone")
WORK = Path("/content/piper-work")
DRIVE_ROOT = Path("/content/drive/MyDrive/piper-voice-clone")
sys.path.insert(0, str(ROOT / "src"))

from voice_cloning.dataset_generator import dataset_dir_name, load_config
from voice_cloning.dataset_import import import_dataset_archive

cfg = load_config(ROOT / "config.colab.yaml")
dataset_name = dataset_dir_name(
    cfg.voice_name, cfg.language_code, cfg.dataset_version
)
default_drive_zip = DRIVE_ROOT / "archives" / f"{dataset_name}.zip"

output_root = WORK / "datasets"
dataset_dir = output_root / dataset_name

if not IMPORT_FROM_DRIVE:
    print("Skipped dataset import — run cell 10 to generate, or enable IMPORT_FROM_DRIVE.")
else:
    zip_path = Path(DRIVE_ZIP_PATH.strip() or default_drive_zip)
    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Dataset zip not found: {zip_path}\n"
            "Mount Drive (cell 3), publish a dataset archive first, or set DRIVE_ZIP_PATH."
        )
    import_dataset_archive(
        zip_path,
        output_root=output_root,
        dataset_name=dataset_name,
        force=FORCE_IMPORT,
    )
    meta = dataset_dir / "metadata.csv"
    if not meta.is_file():
        meta = dataset_dir / "metadata_train.csv"
    if meta.is_file():
        print("Imported dataset:", dataset_dir)
        print("Metadata:", meta)
    else:
        print("Warning: import finished but no metadata.csv found under", dataset_dir)

### Generate dataset (cell 10)

Synthesizes TTS utterances with Qwen3-TTS from your reference voice (cell 5).  
Output: `/content/piper-work/datasets/{voice}-{lang}_{version}/`

**Skip** if you imported a complete dataset in cell 9. With resume enabled, re-running adds more WAVs to an existing dataset.

| Parameter | What it does |
|-----------|--------------|
| **NUM_UTTERANCES** | How many utterances to generate. Default **10** — listen to a few samples and check voice quality. Change to **2,000–2,500** for real training. Re-run the cell to continue (resume keeps existing WAVs). |
| **TEMPERATURE** | TTS sampling temperature (default **0.9**). Lower = more stable; higher = more varied. |
| **SKIP_IF_EXISTS** | Skip when `metadata.csv` already exists. |
| **FORCE_REWRITE** | Checked: regenerate all utterances (overwrite existing WAVs). Unchecked (default): resume and skip existing files. Overrides **SKIP_IF_EXISTS**. |
| **SKIP_DRIVE_PUBLISH** | Checked: keep dataset on Colab disk only. Unchecked (default): publish zip to Google Drive when done. |

Progress prints one line per utterance. Listen to samples in **cell 11**.

In [ ]:
# @title 10. Generate dataset
NUM_UTTERANCES = 10  # @param {type:"integer"}
TEMPERATURE = 0.9  # @param {type:"number"}
SKIP_IF_EXISTS = True  # @param {type:"boolean"}
FORCE_REWRITE = False  # @param {type:"boolean"}
SKIP_DRIVE_PUBLISH = False  # @param {type:"boolean"}

import sys
from pathlib import Path

ROOT = Path("/content/piper-voice-clone")
WORK = Path("/content/piper-work")
sys.path.insert(0, str(ROOT / "src"))
from voice_cloning.dataset_generator import load_config

cfg = load_config(ROOT / "config.colab.yaml")
dataset_dir = WORK / "datasets" / f"{cfg.voice_name}-{cfg.language_code}_{cfg.dataset_version}"
has_dataset = (dataset_dir / "metadata.csv").is_file() or (
    dataset_dir / "metadata_train.csv"
).is_file()

if SKIP_IF_EXISTS and has_dataset and not FORCE_REWRITE:
    print("Dataset already exists, skipping generation:", dataset_dir)
else:
    %cd /content/piper-voice-clone
    if NUM_UTTERANCES < 2000:
        print(
            f"Generating {NUM_UTTERANCES} utterances. "
            "Set NUM_UTTERANCES to 2000–2500 for training, then re-run."
        )
    else:
        print(f"Generating {NUM_UTTERANCES} utterances...")
    if FORCE_REWRITE:
        print("Force rewrite enabled — regenerating all utterances.")
    print(f"TTS temperature: {TEMPERATURE}")
    publish_flag = " --skip-publish" if SKIP_DRIVE_PUBLISH else ""
    rewrite_flag = " --force-rewrite" if FORCE_REWRITE else ""
    !bash colab/generate_dataset.sh --config config.colab.yaml --samples {NUM_UTTERANCES} --temperature {TEMPERATURE}{rewrite_flag}{publish_flag}
    print("Dataset ready:", dataset_dir)

### Preview dataset samples (cell 11)

Listen to random utterances from your dataset after **cell 9** (import) or **cell 10** (generate).

| Parameter | What it does |
|-----------|--------------|
| **PREVIEW_COUNT** | How many random samples to play (default **10**). Capped at the number of WAVs available. |
| **PREVIEW_SEED** | Random seed for reproducible selection. |

Each sample shows the transcript and an audio player.

In [ ]:
# @title 11. Preview dataset samples
PREVIEW_COUNT = 10  # @param {type:"integer"}
PREVIEW_SEED = 42  # @param {type:"integer"}

import sys
from pathlib import Path

from IPython.display import Audio, Markdown, display

ROOT = Path("/content/piper-voice-clone")
WORK = Path("/content/piper-work")
sys.path.insert(0, str(ROOT / "src"))

from voice_cloning.dataset_generator import load_config
from voice_cloning.metadata import sample_dataset_previews

cfg = load_config(ROOT / "config.colab.yaml")
dataset_dir = WORK / "datasets" / f"{cfg.voice_name}-{cfg.language_code}_{cfg.dataset_version}"

samples, available = sample_dataset_previews(
    dataset_dir,
    count=PREVIEW_COUNT,
    seed=PREVIEW_SEED,
)

if PREVIEW_COUNT > available:
    print(
        f"Requested {PREVIEW_COUNT} samples but only {available} available — playing all {len(samples)}."
    )
else:
    print(f"Playing {len(samples)} random sample(s) from {available} utterances in {dataset_dir}")

for index, sample in enumerate(samples, start=1):
    display(Markdown(f"**{index}/{len(samples)}** — {sample.text}"))
    display(Audio(filename=str(sample.wav_path)))

In [ ]:
# @title 12. TensorBoard (run before / during training)
from pathlib import Path
import sys

PROJECT = Path("/content/piper-voice-clone")
WORK = Path("/content/piper-work")

sys.path.insert(0, str(PROJECT / "src"))
from voice_cloning.dataset_generator import load_config

cfg = load_config(PROJECT / "config.colab.yaml")
LOCAL_LIGHTNING = WORK / "lightning_logs" / cfg.voice_name

sys.path.insert(0, str(PROJECT / "colab"))
from piper_training_utils import resolve_tensorboard_logdir

TB_LOGDIR = resolve_tensorboard_logdir(LOCAL_LIGHTNING)
print("TensorBoard logdir:", TB_LOGDIR)

%load_ext tensorboard
%tensorboard --logdir {TB_LOGDIR} --reload_interval 30

### Train (cell 13)

Fine-tune Piper on your dataset. Uses voice / language / dataset from **cell 4**.  
Watch `val_loss` in TensorBoard (**cell 12**) while training.

| Parameter | What it does |
|-----------|--------------|
| **ADD_EPOCHS** | Add this many epochs to the current checkpoint (default 10). If no checkpoint exists, trains for this many epochs from pretrained. |
| **TRAIN_BATCH_SIZE** | Samples per step; **0** = auto from GPU VRAM (16 on L4/T4). |
| **SKIP_DRIVE_PUBLISH** | Checked: keep ONNX/checkpoints on Colab disk only (not copied to Drive). |
| **KEEP_RUNTIME_CONNECTED** | Checked: stay connected after training (use anti-disconnect, cell 7). |

~2,000 samples on L4: epoch 1 ≈ **30 min** (cache warmup), then ≈ **1 min/epoch**. Re-run cell 2 after repo updates.

In [ ]:
# @title 13. Train
ADD_EPOCHS = 10  # @param {type:"integer"}
TRAIN_BATCH_SIZE = 0  # @param {type:"integer"}
SKIP_DRIVE_PUBLISH = False  # @param {type:"boolean"}
KEEP_RUNTIME_CONNECTED = False  # @param {type:"boolean"}

import sys
from pathlib import Path

ROOT = Path("/content/piper-voice-clone")
WORK = Path("/content/piper-work")
sys.path.insert(0, str(ROOT / "colab"))
sys.path.insert(0, str(ROOT / "src"))

from piper_training_utils import colab_training_profile
from voice_cloning.dataset_generator import load_config

cfg = load_config(ROOT / "config.colab.yaml")
dataset_dir = WORK / "datasets" / f"{cfg.voice_name}-{cfg.language_code}_{cfg.dataset_version}"
profile = colab_training_profile(batch_size=TRAIN_BATCH_SIZE)

extra_flags = ""
if TRAIN_BATCH_SIZE > 0:
    extra_flags += f" --batch {TRAIN_BATCH_SIZE}"
if SKIP_DRIVE_PUBLISH:
    extra_flags += " --skip-publish"
if KEEP_RUNTIME_CONNECTED:
    extra_flags += " --no-disconnect"

print(
    f"""
Training configuration
──────────────────────
  voice / dataset   {VOICE_NAME}-{LANGUAGE_CODE}_{DATASET_VERSION}
                    {dataset_dir}
  pretrained base   {CHECKPOINT_PRESET if CHECKPOINT_PRESET != "custom" else EFFECTIVE_CHECKPOINT_HF_PATH}
                    /content/piper-work/checkpoints/pretrained-finetune.ckpt
  add epochs        {ADD_EPOCHS}
  batch size        {profile['batch_size']} ({'auto' if TRAIN_BATCH_SIZE <= 0 else TRAIN_BATCH_SIZE})
  precision         {profile['precision']}  (16-bit mixed; required for Piper STFT on Colab)
  data workers      {profile['num_workers']}
  GPU               {profile['gpu_name']} ({profile['vram_gb']} GB)
  validation        every epoch, up to {profile['limit_val_batches']} batches
  Drive publish     {'off (local only)' if SKIP_DRIVE_PUBLISH else 'on at end + every 10 epochs'}
  disconnect after  {'no (--no-disconnect)' if KEEP_RUNTIME_CONNECTED else 'yes (releases GPU)'}
──────────────────────
~2000 samples on L4: epoch 1 ≈ 30 min (cache warmup), then ≈ 1 min/epoch.
Re-run cell 2 after repo updates so Piper patches apply.
"""
)

%cd /content/piper-voice-clone
!bash colab/train_piper.sh --voice {VOICE_NAME} --lang {LANGUAGE_CODE} --version {DATASET_VERSION} --add-epochs {ADD_EPOCHS}{extra_flags}